# Combo #3: AdaBN-ChakraNet
## Test-Time Adaptive Batch Normalization for Cross-Hospital Generalization & Covariate Shift Mitigation

---

### 1. Clinical Context & The Challenge of Domain Shift in Endoscopy
Colorectal cancer (CRC) is the third most commonly diagnosed malignancy and the second leading cause of cancer mortality worldwide. Routine screening via optical colonoscopy enables early detection and resection of adenomatous polyps, drastically reducing CRC incidence. However, deep neural networks trained on colonoscopy datasets from a single medical center frequently suffer catastrophic performance degradation ($8\% - 18\%$ drop in Dice Similarity Coefficient) when deployed across external hospital networks.

This phenomenon, known as **Cross-Center Domain Shift** (or **Covariate Shift**), arises from substantial inter-center heterogeneity:
1. **Endoscope Hardware & Sensor Discrepancies**: Different manufacturers (e.g., Olympus Exera III / Lucera Elite, Pentax Medical OPTIVISTA EPK-i7010, Fujifilm ELUXEO 7000) utilize proprietary CCD/CMOS sensors with vastly distinct dynamic ranges, chromatic fidelity, and Bayer filter arrays.
2. **Illumination Optics & Chromoendoscopy Modes**: Xenon arc bulbs vs. multi-LED optical sources produce divergent color temperatures ($3000\text{K} - 6500\text{K}$). Narrow-Band Imaging (NBI), Flexible Spectral Imaging Color Enhancement (FICE), and blue-laser imaging skew color distributions toward green-cyan or deep magenta.
3. **Multi-Center Cohort Characteristics**:
   - **Kvasir-SEG (Bærum Hospital, Norway)**: Standard white-light colonoscopy, high saturation Olympus scopes, diverse lesion morphologies.
   - **CVC-ClinicDB (Hospital Clínic Barcelona, Spain)**: High mucosal moisture, Pentax high-definition scopes, elevated green-channel bile artifacts.
   - **ETIS-Larib (Lariboisière Hospital, France)**: Low-light, high-gain video frames with subtle, flat polyps (Paris classification IIa/IIb).

```
  Source Center (Norway / Kvasir-SEG)          Target Center (Spain / CVC-ClinicDB)
 ┌──────────────────────────────────────┐     ┌──────────────────────────────────────┐
 │ • High Saturation (Olympus)          │     │ • Pentax HD Sensor / Green Bile Tone │
 │ • P_S(X) Feature Distribution        │     │ • P_T(X) Shifted Feature Distribution│
 └──────────────────┬───────────────────┘     └──────────────────┬───────────────────┘
                    │                                            │
                    ▼                                            ▼
       ┌────────────────────────┐                   ┌────────────────────────┐
       │ Learned Weights: W, b  │                   │ Learned Weights: W, b  │ (Unchanged!)
       │ Running Mean:   μ_S    │ ═════════════════►│ Running Mean:   μ_T    │ (Adapted via AdaBN)
       │ Running Var:    σ²_S   │  Test-Time AdaBN  │ Running Var:    σ²_T   │ (Adapted via AdaBN)
       └────────────────────────┘                   └────────────────────────┘
```

---

### 2. Mathematical Formulation of Covariate Shift & AdaBN
Let $\mathcal{X} \subset \mathbb{R}^{H \times W \times 3}$ denote the endoscopic input space and $\mathcal{Y} \subset \{0, 1\}^{H \times W}$ denote the binary segmentation space.
Under the **Covariate Shift Assumption**:
$$P_S(X) \neq P_T(X) \quad \text{while} \quad P(Y|X) \text{ remains invariant}$$

Standard Convolutional Neural Networks normalize intermediate feature representations using Batch Normalization layers:
$$\hat{\mathbf{z}}^{(l)} = \frac{\mathbf{z}^{(l)} - \mu^{(l)}}{\sqrt{(\sigma^{(l)})^2 + \epsilon}} \odot \gamma^{(l)} + \beta^{(l)}$$

During training on the source domain $\mathcal{D}_S$, the layer computes and accumulates exponential moving averages:
$$\mu_{running}^{(l)} \gets (1 - \alpha) \mu_{running}^{(l)} + \alpha \mu_B^{(l)}, \quad (\sigma^2)_{running}^{(l)} \gets (1 - \alpha) (\sigma^2)_{running}^{(l)} + \alpha (\sigma_B^2)^{(l)}$$

When evaluating on target domain $\mathcal{D}_T$, using the source statistics $(\mu_S, \sigma^2_S)$ leads to severe feature misalignment because:
$$\mathbb{E}_{x \sim P_T}[\mathbf{z}^{(l)}] \neq \mu_S^{(l)}, \quad \text{Var}_{x \sim P_T}[\mathbf{z}^{(l)}] \neq (\sigma_S^2)^{(l)}$$

#### The Adaptive Batch Normalization (AdaBN) Solution
AdaBN demonstrates that **domain-specific characteristics (illumination, sensor color palette, gain) are primarily encoded in the normalization statistics ($\mu, \sigma^2$), whereas domain-invariant structural features (mucosal edges, polyp geometry) are captured by the convolution weights ($W, b, \gamma, \beta$)**.

AdaBN executes zero-shot test-time adaptation as follows:
1. **Freeze all learnable parameters**: $\theta = \{W_l, b_l, \gamma_l, \beta_l\}_{l=1}^L$ are completely locked.
2. **Reset running statistics**:
   $$\mu_{running}^{(l)} \gets \mathbf{0}, \quad (\sigma^2)_{running}^{(l)} \gets \mathbf{1}, \quad \text{momentum} \gets \text{None} \text{ (cumulative mean)}$$
3. **Stream unlabelled target domain batches** $\mathbf{X}_T \sim \mathcal{D}_T$ in forward-only mode (`torch.no_grad()`):
   $$\mu_{T}^{(l)} = \frac{1}{N \cdot H \cdot W} \sum_{n=1}^N \sum_{h=1}^H \sum_{w=1}^W \mathbf{z}_{n, :, h, w}^{(l)}$$
   $$(\sigma_T^2)^{(l)} = \frac{1}{N \cdot H \cdot W} \sum_{n=1}^N \sum_{h=1}^H \sum_{w=1}^W \left( \mathbf{z}_{n, :, h, w}^{(l)} - \mu_T^{(l)} \right)^2$$
4. **Lock adapted statistics** and execute inference.

**Key Advantages**:
- **Zero Backpropagation**: Adaptation requires only forward passes; memory consumption is minimal.
- **Zero Label Requirement**: 100% unsupervised test-time adaptation.
- **Zero Catastrophic Forgetting**: Convolutional filter weights remain pristine.

---

### 3. PraNet ResNet-101 Backbone Architecture
This notebook integrates the upgraded **PraNetResNet101** architecture:
- **ResNet-101 Backbone**: 44.5M parameters with ImageNet-1K V2 weights across 4 residual stages.
- **Receptive Field Blocks (RFB 1-4)**: Multi-branch dilated atrous convolutions ($d=3, 5, 7$) capturing multi-scale context.
- **Parallel Partial Decoder (PPD)**: Aggregates semantic features ($\mathbf{R}_2, \mathbf{R}_3, \mathbf{R}_4$) to generate global coarse saliency $S_g$.
- **Cascaded Reverse Attention (RA 1-4) with CBAM**: Inverts saliency maps ($1 - \sigma(S)$) to iteratively refine polyp boundaries from coarse to fine resolution.
- **Deep Supervision**: Multi-level loss backpropagation across all 4 RA stages and global saliency.

---

### 4. Maximum Hardware Execution Specifications
- **Batch Size**: `32` per GPU worker
- **DataLoader Multiprocessing**: `num_workers = 4`, `pin_memory = True`
- **Mixed Precision**: Automatic Mixed Precision (AMP FP16) with `torch.cuda.amp.GradScaler`
- **CuDNN Optimization**: `torch.backends.cudnn.benchmark = True`, TF32 enabled


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT, DEPENDENCIES & HARDWARE VERIFICATION
# ==============================================================================

import os
import sys
import time
import random
import subprocess
import numpy as np
import torch
import torchvision

# Verify & Install Dependencies
required_packages = ["albumentations", "opencv-python-headless", "matplotlib", "seaborn", "tqdm", "scipy"]
for pkg in required_packages:
    try:
        __import__(pkg.split("-")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

# Hardware Acceleration & CuDNN Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Active Device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 Matrix Multiplication & CuDNN Benchmark: ENABLED")
else:
    print("Running in CPU mode. (CUDA recommended for max performance)")

# Set deterministic seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Reproducibility Seed set to 42.")


In [ ]:
# ==============================================================================
# CELL 2: SYSTEM CONFIGURATION & MAX-SPEC HYPERPARAMETERS
# ==============================================================================

from dataclasses import dataclass

@dataclass
class AdaBNConfig:
    # Hardware & Multiprocessing
    batch_size: int = 32
    num_workers: int = 4
    pin_memory: bool = True
    image_size: tuple = (352, 352)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Source Training Hyperparameters
    epochs: int = 25
    lr: float = 1e-3
    backbone_lr_ratio: float = 0.1
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    
    # AdaBN Domain Adaptation Parameters
    n_adapt_batches: int = 10     # Number of unlabelled target batches for stats recalibration
    adapt_momentum: float = None  # None indicates cumulative average over target stream
    
    # Synthetic / Simulated Multi-Center Domain Shift Parameters
    target_color_temp: float = 0.85     # Shift color balance toward cold/greenish endoscopic light
    target_bile_cast: float = 0.20       # Green bile tint augmentation for CVC-ClinicDB emulation
    target_specular_gain: float = 1.30   # Elevated specular wet reflection intensity
    target_noise_sigma: float = 12.0     # Elevated sensor read noise

config = AdaBNConfig()
print("Max-Spec AdaBN-ChakraNet Configuration:")
for k, v in vars(config).items():
    print(f"  * {k:22s}: {v}")


In [ ]:
# ==============================================================================
# CELL 3: AUTOMATED DATASET ACQUISITION & VERIFICATION (KVASIR-SEG)
# Self-contained, robust download and layout normalization for Kaggle/Colab/Local
# ==============================================================================

import shutil
import zipfile
import ssl

def setup_kvasir_seg_dataset(
    target_dir=None,
    force_download=False,
    synthetic_fallback_count=1000,
    verbose=True
):
    # 1. Resolve Environment and Target Directory
    if target_dir is not None:
        dataset_dir = Path(target_dir).resolve()
    elif Path("/kaggle/working").exists():
        dataset_dir = Path("/kaggle/working/data/kvasir-seg")
    elif Path("/content").exists():
        dataset_dir = Path("/content/data/kvasir-seg")
    else:
        dataset_dir = Path("./data/kvasir-seg").resolve()

    images_dir = dataset_dir / "images"
    masks_dir = dataset_dir / "masks"
    images_dir.mkdir(parents=True, exist_ok=True)
    masks_dir.mkdir(parents=True, exist_ok=True)

    def log(msg):
        if verbose:
            print(f"[Kvasir-SEG Pipeline] {msg}")

    log(f"Configured dataset directory: {dataset_dir}")

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    
    def validate_pairs(img_d, msk_d):
        if not img_d.exists() or not msk_d.exists():
            return False, 0, 0
        imgs = {p.stem: p for p in img_d.glob("*") if p.suffix.lower() in valid_exts}
        msks = {p.stem: p for p in msk_d.glob("*") if p.suffix.lower() in valid_exts}
        common = set(imgs.keys()).intersection(set(msks.keys()))
        is_ok = len(imgs) >= 1000 and len(msks) >= 1000 and len(common) == len(imgs)
        return is_ok, len(imgs), len(msks)

    if not force_download:
        is_valid, n_img, n_msk = validate_pairs(images_dir, masks_dir)
        if is_valid:
            log(f"Verified existing dataset: {n_img} images, {n_msk} masks. Ready!")
            return dataset_dir

    # Check /kaggle/input for mounted datasets
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        log("Checking /kaggle/input for mounted datasets...")
        candidates = [
            d for d in kaggle_input.rglob("*")
            if d.is_dir() and d.name.lower() in {"kvasir-seg", "kvasirseg", "kvasir_seg"}
        ]
        for c_dir in candidates:
            c_imgs = c_dir / "images" if (c_dir / "images").exists() else c_dir / "Images"
            c_msks = c_dir / "masks" if (c_dir / "masks").exists() else c_dir / "Masks"
            if c_imgs.exists() and c_msks.exists():
                log(f"Found mounted dataset in {c_dir}. Copying files...")
                for f in c_imgs.glob("*"):
                    if f.is_file() and f.suffix.lower() in valid_exts:
                        shutil.copy2(f, images_dir / f.name)
                for f in c_msks.glob("*"):
                    if f.is_file() and f.suffix.lower() in valid_exts:
                        shutil.copy2(f, masks_dir / f.name)
                is_valid, n_img, n_msk = validate_pairs(images_dir, masks_dir)
                if is_valid or n_img >= 1000:
                    log(f"Copied from Kaggle input: {n_img} images, {n_msk} masks.")
                    return dataset_dir

    # Download Cascade
    download_urls = [
        "https://datasets.simula.no/downloads/kvasir-seg.zip",
        "https://huggingface.co/datasets/polyp-segmentation/kvasir-seg/resolve/main/kvasir-seg.zip",
        "https://zenodo.org/record/4646797/files/kvasir-seg.zip"
    ]

    zip_dest = dataset_dir.parent / "kvasir-seg-download.zip"
    download_success = False

    def download_requests(url, dest):
        try:
            import requests
            import urllib3
            urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
            log(f"Connecting to {url}...")
            with requests.get(url, stream=True, timeout=(15, 90), verify=False) as resp:
                if resp.status_code != 200:
                    return False
                total_size = int(resp.headers.get("content-length", 0))
                downloaded = 0
                with open(dest, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            if total_size > 0:
                                pct = downloaded / total_size * 100
                                print(f"\r  Progress: {pct:5.1f}% ({downloaded//1048576}MB / {total_size//1048576}MB)", end="", flush=True)
                print()
                return dest.exists() and dest.stat().st_size > 1_000_000
        except Exception as e:
            log(f"Requests error: {e}")
            return False

    def download_urllib(url, dest):
        try:
            import urllib.request
            ctx = ssl.create_default_context()
            ctx.check_hostname = False
            ctx.verify_mode = ssl.CERT_NONE
            log(f"Connecting via urllib to {url}...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=ctx, timeout=60) as resp, open(dest, "wb") as f:
                total_size = int(resp.headers.get("content-length", 0))
                downloaded = 0
                while True:
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        pct = downloaded / total_size * 100
                        print(f"\r  Progress: {pct:5.1f}% ({downloaded//1048576}MB)", end="", flush=True)
            print()
            return dest.exists() and dest.stat().st_size > 1_000_000
        except Exception as e:
            log(f"Urllib error: {e}")
            return False

    for url in download_urls:
        log(f"Attempting download: {url}")
        for attempt in range(1, 3):
            if download_requests(url, zip_dest) or download_urllib(url, zip_dest):
                download_success = True
                log(f"Download successful: {zip_dest.stat().st_size / (1024*1024):.2f} MB")
                break
            log(f"Retry {attempt+1}/2...")
        if download_success:
            break

    # Extract & Normalize
    if download_success and zip_dest.exists():
        temp_extract = dataset_dir.parent / "_kvasir_raw_temp"
        log("Extracting and organizing dataset files...")
        try:
            with zipfile.ZipFile(zip_dest, "r") as zf:
                zf.extractall(temp_extract)

            found_imgs = []
            found_masks = []
            for file_path in temp_extract.rglob("*"):
                if file_path.is_file() and file_path.suffix.lower() in valid_exts:
                    parts = [p.lower() for p in file_path.parts]
                    if any(m in parts for m in ["mask", "masks", "ground_truth", "gt"]):
                        found_masks.append(file_path)
                    elif any(img in parts for img in ["image", "images", "original"]):
                        found_imgs.append(file_path)

            for p in found_imgs:
                shutil.copy2(p, images_dir / p.name)
            for p in found_masks:
                shutil.copy2(p, masks_dir / p.name)

            log(f"Extracted {len(found_imgs)} images and {len(found_masks)} masks.")
        except Exception as e:
            log(f"Extraction error: {e}")
        finally:
            shutil.rmtree(temp_extract, ignore_errors=True)
            if zip_dest.exists():
                zip_dest.unlink(missing_ok=True)

    is_valid, n_img, n_msk = validate_pairs(images_dir, masks_dir)
    if is_valid or (n_img >= 1000 and n_msk >= 1000):
        log(f"Dataset successfully prepared: {n_img} images, {n_msk} masks.")
        return dataset_dir

    # Synthetic Generator Fallback
    log(f"Notice: Download incomplete ({n_img}/1000 images). Generating synthetic endoscopic dataset...")
    np.random.seed(42)
    h, w = 352, 352
    needed = max(0, synthetic_fallback_count - n_img)

    for i in range(needed):
        img = np.full((h, w, 3), (np.random.randint(40, 70), np.random.randint(60, 100), np.random.randint(140, 190)), dtype=np.uint8)
        noise = np.random.normal(0, 10, (h, w, 3)).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        y, x = np.ogrid[:h, :w]
        dist = np.sqrt((x - w/2)**2 + (y - h/2)**2)
        vignette = 1.0 - 0.35 * (dist / np.sqrt((w/2)**2 + (h/2)**2))**1.5
        img = np.clip(img * vignette[..., None], 0, 255).astype(np.uint8)

        mask = np.zeros((h, w), dtype=np.uint8)
        if np.random.rand() > 0.05:
            cx, cy = np.random.randint(int(w*0.25), int(w*0.75)), np.random.randint(int(h*0.25), int(h*0.75))
            rx, ry = np.random.randint(int(w*0.10), int(w*0.25)), np.random.randint(int(h*0.08), int(h*0.22))
            angle = np.random.randint(0, 180)
            cv2.ellipse(mask, (cx, cy), (rx, ry), angle, 0, 360, 255, -1)
            polyp_bgr = (np.random.randint(30, 60), np.random.randint(45, 85), np.random.randint(170, 230))
            cv2.ellipse(img, (cx, cy), (rx, ry), angle, 0, 360, polyp_bgr, -1)
            if np.random.rand() > 0.3:
                cv2.circle(img, (cx + np.random.randint(-5, 5), cy + np.random.randint(-5, 5)), np.random.randint(3, 7), (240, 245, 255), -1)

        file_id = f"cju_syn_{i:04d}"
        cv2.imwrite(str(images_dir / f"{file_id}.jpg"), img)
        cv2.imwrite(str(masks_dir / f"{file_id}.jpg"), mask)

    is_valid, n_img, n_msk = validate_pairs(images_dir, masks_dir)
    log(f"Ready: {n_img} images and {n_msk} masks located in {dataset_dir}")
    return dataset_dir

DATASET_PATH = setup_kvasir_seg_dataset()
print(f"Kvasir-SEG Dataset root: {DATASET_PATH}")


In [ ]:
# ==============================================================================
# CELL 4: DATASET LOADERS & MULTI-CENTER DOMAIN SHIFT SIMULATION
# Source (Norway) vs Target (Spain / CVC-ClinicDB Simulation)
# ==============================================================================

from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif'}

class MaxSpecPolypDataset(Dataset):
    # Source Domain Dataset (Kvasir-SEG standard distribution)
    def __init__(self, img_dir, mask_dir, size=352, augment=True):
        self.imgs = sorted([p for p in Path(img_dir).glob('*') if p.suffix.lower() in IMAGE_EXTS])
        self.mask_dir = Path(mask_dir)
        self.size = size
        self.augment = augment
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        self.jitter = T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08)

    def __len__(self):
        return len(self.imgs)

    def _find_mask(self, stem):
        for ext in IMAGE_EXTS:
            mp = self.mask_dir / (stem + ext)
            if mp.exists():
                return mp
        return None

    def __getitem__(self, idx):
        ip = self.imgs[idx]
        mp = self._find_mask(ip.stem)
        img = cv2.imread(str(ip))
        mask = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) if mp else None

        if img is None:
            img = np.zeros((self.size, self.size, 3), np.uint8)
        if mask is None:
            mask = np.zeros((self.size, self.size), np.uint8)

        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

        if self.augment:
            if np.random.rand() > 0.5:
                img, mask = cv2.flip(img, 1), cv2.flip(mask, 1)
            if np.random.rand() > 0.5:
                img, mask = cv2.flip(img, 0), cv2.flip(mask, 0)
            angle = np.random.uniform(-25, 25)
            M = cv2.getRotationMatrix2D((self.size // 2, self.size // 2), angle, 1.0)
            img = cv2.warpAffine(img, M, (self.size, self.size))
            mask = cv2.warpAffine(mask, M, (self.size, self.size))

        img_t = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0
        if self.augment:
            img_t = self.jitter(img_t)
            if np.random.rand() > 0.5:
                img_t = torch.clamp(img_t + torch.randn_like(img_t) * 0.015, 0.0, 1.0)

        img_t = (img_t - self.mean) / self.std
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


class TargetDomainShiftDataset(Dataset):
    # Target Domain Dataset (Emulating CVC-ClinicDB / ETIS-Larib sensor and optical shift)
    def __init__(self, img_dir, mask_dir, size=352, unlabelled=False):
        self.imgs = sorted([p for p in Path(img_dir).glob('*') if p.suffix.lower() in IMAGE_EXTS])
        self.mask_dir = Path(mask_dir)
        self.size = size
        self.unlabelled = unlabelled
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.imgs)

    def _find_mask(self, stem):
        for ext in IMAGE_EXTS:
            mp = self.mask_dir / (stem + ext)
            if mp.exists():
                return mp
        return None

    def _apply_clinical_domain_shift(self, bgr_img):
        # 1. Color balance shift (Pentax-style cooler sensor + bile greenish hue)
        img_f = bgr_img.astype(np.float32)
        b, g, r = cv2.split(img_f)
        b = np.clip(b * 1.15, 0, 255)
        g = np.clip(g * 1.25 + 10.0, 0, 255)
        r = np.clip(r * 0.85, 0, 255)
        shifted = cv2.merge([b, g, r])

        # 2. Non-linear Gamma distortion (optical intensity compression)
        gamma = 1.35
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        shifted = cv2.LUT(np.clip(shifted, 0, 255).astype(np.uint8), table)

        # 3. High-gain sensor noise
        noise = np.random.normal(0, 8.0, shifted.shape).astype(np.int16)
        shifted = np.clip(shifted.astype(np.int16) + noise, 0, 255).astype(np.uint8)

        # 4. Specular glare reflections
        if np.random.rand() > 0.4:
            h, w = shifted.shape[:2]
            cx, cy = np.random.randint(w//4, 3*w//4), np.random.randint(h//4, 3*h//4)
            cv2.circle(shifted, (cx, cy), np.random.randint(6, 14), (250, 255, 255), -1)

        return shifted

    def __getitem__(self, idx):
        ip = self.imgs[idx]
        img = cv2.imread(str(ip))
        if img is None:
            img = np.zeros((self.size, self.size, 3), np.uint8)
        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        img = self._apply_clinical_domain_shift(img)

        img_t = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0
        img_t = (img_t - self.mean) / self.std

        if self.unlabelled:
            return img_t

        mp = self._find_mask(ip.stem)
        mask = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) if mp else None
        if mask is None:
            mask = np.zeros((self.size, self.size), np.uint8)
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


# Build Source & Target Partitions
img_dir = DATASET_PATH / "images"
mask_dir = DATASET_PATH / "masks"

all_stems = sorted([p.stem for p in img_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS])
n_total = len(all_stems)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

src_train_ds = Subset(MaxSpecPolypDataset(img_dir, mask_dir, size=352, augment=True), range(0, n_train))
src_val_ds   = Subset(MaxSpecPolypDataset(img_dir, mask_dir, size=352, augment=False), range(n_train, n_train + n_val))
tgt_adapt_ds = Subset(TargetDomainShiftDataset(img_dir, mask_dir, size=352, unlabelled=True), range(n_train + n_val, n_total))
tgt_test_ds  = Subset(TargetDomainShiftDataset(img_dir, mask_dir, size=352, unlabelled=False), range(n_train + n_val, n_total))

source_train_loader = DataLoader(
    src_train_ds, batch_size=config.batch_size, shuffle=True,
    num_workers=config.num_workers, pin_memory=config.pin_memory, drop_last=True
)
source_val_loader = DataLoader(
    src_val_ds, batch_size=config.batch_size, shuffle=False,
    num_workers=config.num_workers, pin_memory=config.pin_memory
)
target_unlabelled_loader = DataLoader(
    tgt_adapt_ds, batch_size=config.batch_size, shuffle=True,
    num_workers=config.num_workers, pin_memory=config.pin_memory, drop_last=False
)
target_test_loader = DataLoader(
    tgt_test_ds, batch_size=config.batch_size, shuffle=False,
    num_workers=config.num_workers, pin_memory=config.pin_memory
)

print(f"Source Train Cohort:      {len(src_train_ds)} samples ({len(source_train_loader)} batches @ BS={config.batch_size})")
print(f"Source Val Cohort:        {len(src_val_ds)} samples ({len(source_val_loader)} batches)")
print(f"Target Unlabelled Stream: {len(tgt_adapt_ds)} samples ({len(target_unlabelled_loader)} batches)")
print(f"Target Evaluation Cohort: {len(tgt_test_ds)} samples ({len(target_test_loader)} batches)")


In [ ]:
# ==============================================================================
# CELL 5: FULL PRANET-RESNET101 ARCHITECTURE
# Upgraded with 4-Stage RFB, PPD Global Decoder, CBAM Reverse Attention & Deep Supervision
# ==============================================================================

import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class BasicConv2d(nn.Module):
    # Standard Convolution-BatchNorm-ReLU Block
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1, relu=True):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(
            in_planes, out_planes,
            kernel_size=kernel_size, stride=stride,
            padding=padding, dilation=dilation, bias=False
        )
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True) if relu else nn.Identity()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class RFBBlock(nn.Module):
    # Receptive Field Block with multi-dilation atrous convolutions
    def __init__(self, in_channel, out_channel):
        super(RFBBlock, self).__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), 1))
        return self.relu(x_cat + self.conv_res(x))


class CBAM(nn.Module):
    # Convolutional Block Attention Module: Channel Attention + Spatial Attention
    def __init__(self, channels, r=8):
        super(CBAM, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // r, channels, bias=False)
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        ca = torch.sigmoid(avg_out + max_out).view(x.size(0), -1, 1, 1)
        x = x * ca
        avg_s = torch.mean(x, dim=1, keepdim=True)
        max_s, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.sigmoid(self.spatial_conv(torch.cat([avg_s, max_s], dim=1)))
        return x * sa


class ReverseAttention(nn.Module):
    # Reverse Attention Module with Saliency Inversion and CBAM
    def __init__(self, in_channel, out_channel):
        super(ReverseAttention, self).__init__()
        self.conv1 = BasicConv2d(in_channel, out_channel, 3, padding=1)
        self.conv2 = BasicConv2d(out_channel, out_channel, 3, padding=1)
        self.cbam  = CBAM(out_channel)
        self.conv_out = nn.Conv2d(out_channel, 1, kernel_size=1)

    def forward(self, feat, saliency_map):
        rev_weight = 1.0 - torch.sigmoid(saliency_map)
        x = feat * rev_weight.expand_as(feat)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.cbam(x)
        return self.conv_out(x)


class PraNetResNet101(nn.Module):
    # Max-Spec PraNet with ResNet-101 Backbone
    def __init__(self, channels=64, mc_dropout_p=0.15):
        super(PraNetResNet101, self).__init__()
        self.channels = channels
        self.mc_dropout_enabled = False
        self.mc_p = mc_dropout_p

        # Backbone: Pretrained ResNet-101 (ImageNet V2 weights)
        weights = models.ResNet101_Weights.IMAGENET1K_V2 if hasattr(models, 'ResNet101_Weights') else True
        resnet = models.resnet101(weights=weights)
        
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # [B, 256, H/4, W/4]
        self.layer2 = resnet.layer2  # [B, 512, H/8, W/8]
        self.layer3 = resnet.layer3  # [B, 1024, H/16, W/16]
        self.layer4 = resnet.layer4  # [B, 2048, H/32, W/32]

        # Multi-scale Receptive Field Blocks
        self.rfb1 = RFBBlock(256, channels)
        self.rfb2 = RFBBlock(512, channels)
        self.rfb3 = RFBBlock(1024, channels)
        self.rfb4 = RFBBlock(2048, channels)

        # Parallel Partial Decoder (PPD)
        self.ppd_conv = BasicConv2d(channels * 3, channels, 3, padding=1)
        self.ppd_out  = nn.Conv2d(channels, 1, kernel_size=1)

        # Reverse Attention Modules (Cascaded Top-Down Refinement)
        self.ra4 = ReverseAttention(channels, channels)
        self.ra3 = ReverseAttention(channels, channels)
        self.ra2 = ReverseAttention(channels, channels)
        self.ra1 = ReverseAttention(channels, channels)

        # Spatial Dropout for MC Uncertainty
        self.drop = nn.Dropout2d(p=mc_dropout_p)

    def enable_mc_dropout(self):
        self.mc_dropout_enabled = True

    def disable_mc_dropout(self):
        self.mc_dropout_enabled = False

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]
        dropout_active = self.training or self.mc_dropout_enabled

        # Backbone Stage Extraction
        x0 = self.stem(x)
        e1 = self.layer1(x0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        # Receptive Field Blocks
        r1 = self.rfb1(e1)
        r2 = self.rfb2(e2)
        r3 = self.rfb3(e3)
        r4 = self.rfb4(e4)

        if dropout_active:
            r1 = self.drop(r1)
            r2 = self.drop(r2)
            r3 = self.drop(r3)
            r4 = self.drop(r4)

        # Parallel Partial Decoder (PPD at H/8)
        sz2 = r2.shape[2:]
        r3_up = F.interpolate(r3, size=sz2, mode='bilinear', align_corners=False)
        r4_up = F.interpolate(r4, size=sz2, mode='bilinear', align_corners=False)
        ppd_feat = self.ppd_conv(torch.cat([r2, r3_up, r4_up], dim=1))
        s_g = self.ppd_out(ppd_feat)

        # Reverse Attention Cascade
        s_g_r4 = F.interpolate(s_g, size=r4.shape[2:], mode='bilinear', align_corners=False)
        s_4 = self.ra4(r4, s_g_r4)

        s_4_r3 = F.interpolate(s_4, size=r3.shape[2:], mode='bilinear', align_corners=False)
        s_3 = self.ra3(r3, s_4_r3)

        s_3_r2 = F.interpolate(s_3, size=r2.shape[2:], mode='bilinear', align_corners=False)
        s_2 = self.ra2(r2, s_3_r2)

        s_2_r1 = F.interpolate(s_2, size=r1.shape[2:], mode='bilinear', align_corners=False)
        s_1 = self.ra1(r1, s_2_r1)

        # Output Logits [B, 1, H, W]
        out = F.interpolate(s_1, size=(h, w), mode='bilinear', align_corners=False)

        if self.training:
            s_g_up = F.interpolate(s_g, size=(h, w), mode='bilinear', align_corners=False)
            s_4_up = F.interpolate(s_4, size=(h, w), mode='bilinear', align_corners=False)
            s_3_up = F.interpolate(s_3, size=(h, w), mode='bilinear', align_corners=False)
            s_2_up = F.interpolate(s_2, size=(h, w), mode='bilinear', align_corners=False)
            return out, s_2_up, s_3_up, s_4_up, s_g_up

        return out


# Test Model Shape & Verification
model = PraNetResNet101(channels=64).to(config.device)
dummy_input = torch.randn(2, 3, 352, 352).to(config.device)

model.eval()
with torch.no_grad():
    eval_out = model(dummy_input)
    print(f"Eval Mode Output Shape: {eval_out.shape} (Expected: [2, 1, 352, 352])")

model.train()
train_out = model(dummy_input)
print(f"Train Mode Deep Supervision Heads: {len(train_out)} tensors (Out, S2, S3, S4, Sg)")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {n_params / 1e6:.2f} Million")


In [ ]:
# ==============================================================================
# CELL 6: LOSS FUNCTIONS & SOURCE DOMAIN TRAINING LOOP (AMP FP16)
# ==============================================================================

import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torchvision.ops import sigmoid_focal_loss

class DiceFocalLoss(nn.Module):
    # Composite Dice + Focal Loss for Imbalanced Endoscopic Polyps
    def __init__(self, alpha=0.25, gamma=2.0, dice_w=0.6, focal_w=0.4):
        super(DiceFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_w = dice_w
        self.focal_w = focal_w

    def _dice_loss(self, logits, targets, smooth=1e-6):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2, 3))
        cardinality = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        return (1.0 - (2.0 * intersection + smooth) / (cardinality + smooth)).mean()

    def forward(self, logits, targets):
        focal = sigmoid_focal_loss(logits, targets, alpha=self.alpha, gamma=self.gamma, reduction='mean')
        dice = self._dice_loss(logits, targets)
        return self.dice_w * dice + self.focal_w * focal


class DeepSupervisionDiceFocalLoss(nn.Module):
    # Hierarchical Deep Supervision across all 4 RA stages and Global Saliency
    def __init__(self):
        super(DeepSupervisionDiceFocalLoss, self).__init__()
        self.criterion = DiceFocalLoss()

    def forward(self, outputs, targets):
        if isinstance(outputs, tuple):
            out, s2, s3, s4, sg = outputs
            return (
                1.00 * self.criterion(out, targets) +
                0.25 * self.criterion(s2, targets) +
                0.20 * self.criterion(s3, targets) +
                0.15 * self.criterion(s4, targets) +
                0.10 * self.criterion(sg, targets)
            )
        return self.criterion(outputs, targets)


def evaluate_model(model, dataloader, device):
    # Compute mean Dice Similarity Coefficient (DSC) and Mean IoU (mIoU)
    model.eval()
    dices, ious = [], []
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with autocast():
                logits = model(imgs)
                if isinstance(logits, tuple):
                    logits = logits[0]
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            for i in range(len(preds)):
                p = preds[i, 0]
                g = masks[i, 0]
                inter = (p * g).sum().item()
                card = p.sum().item() + g.sum().item()
                dice = (2.0 * inter + 1e-6) / (card + 1e-6)
                union = card - inter
                iou = (inter + 1e-6) / (union + 1e-6)
                dices.append(dice)
                ious.append(iou)
    return float(np.mean(dices)), float(np.mean(ious))


# Optimizer with Differential Learning Rates
backbone_keys = ['stem', 'layer1', 'layer2', 'layer3', 'layer4']
backbone_params = [p for n, p in model.named_parameters() if any(k in n for k in backbone_keys)]
head_params     = [p for n, p in model.named_parameters() if not any(k in n for k in backbone_keys)]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': config.lr * config.backbone_lr_ratio},
    {'params': head_params,     'lr': config.lr}
], weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=1e-5)
criterion = DeepSupervisionDiceFocalLoss()
scaler = GradScaler()

print(f"\nLaunching Source Domain Training ({config.epochs} Epochs, BS={config.batch_size}, AMP FP16)...")
best_source_val_dice = 0.0
best_model_weights = None

for epoch in range(1, config.epochs + 1):
    model.train()
    running_loss = 0.0
    
    for imgs, masks in source_train_loader:
        imgs = imgs.to(config.device, non_blocking=True)
        masks = masks.to(config.device, non_blocking=True)
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
    scheduler.step()
    epoch_loss = running_loss / len(source_train_loader)
    
    if epoch % 5 == 0 or epoch == config.epochs:
        val_dice, val_iou = evaluate_model(model, source_val_loader, config.device)
        print(f"Epoch [{epoch:02d}/{config.epochs}] | Loss: {epoch_loss:.4f} | Source Val DSC: {val_dice:.4f} | Source Val mIoU: {val_iou:.4f}")
        if val_dice > best_source_val_dice:
            best_source_val_dice = val_dice
            best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        print(f"Epoch [{epoch:02d}/{config.epochs}] | Loss: {epoch_loss:.4f}")

if best_model_weights is not None:
    model.load_state_dict({k: v.to(config.device) for k, v in best_model_weights.items()})
print(f"\nSource Domain Training Complete. Best Source Val DSC: {best_source_val_dice:.4f}")


In [ ]:
# ==============================================================================
# CELL 7: TEST-TIME ADABN ADAPTATION ALGORITHM
# Zero-Backpropagation BatchNorm Statistics Recalibration
# ==============================================================================

class AdaBNAdapter:
    # Test-Time Adaptive Batch Normalization Engine for PraNet ResNet-101
    def __init__(self, model, device='cuda'):
        self.model = model.to(device)
        self.device = device
        self.source_stats = {}
        self.target_stats = {}
        self._cache_source_statistics()

    def _cache_source_statistics(self):
        # Records source domain running statistics for drift quantification
        for name, m in self.model.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                self.source_stats[name] = {
                    'mean': m.running_mean.clone().detach().cpu(),
                    'var': m.running_var.clone().detach().cpu()
                }

    def adapt_to_target_domain(self, target_unlabelled_loader, n_adapt_batches=10):
        # Recalibrate all BatchNorm2d layers on target hospital stream
        print(f"\n[AdaBN] Initiating Test-Time Adaptation on unlabelled target domain...")
        print(f"[AdaBN] Target Stream: {n_adapt_batches} calibration batches (BS={target_unlabelled_loader.batch_size})")
        
        # 1. Put model in train mode to activate batch statistic computation
        self.model.train()
        
        # 2. Reset running statistics and configure cumulative averaging
        bn_count = 0
        for name, m in self.model.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                m.reset_running_stats()
                m.momentum = None  # Computes simple cumulative average over streaming batches
                bn_count += 1
        print(f"[AdaBN] Reset running statistics across {bn_count} BatchNorm2d layers.")

        # 3. Stream target domain unlabelled batches with zero backpropagation
        start_t = time.time()
        with torch.no_grad():
            for i, batch in enumerate(target_unlabelled_loader):
                if i >= n_adapt_batches:
                    break
                imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
                imgs = imgs.to(self.device, non_blocking=True)
                with autocast():
                    _ = self.model(imgs)  # Forward pass calculates target domain statistics
        
        adaptation_time = time.time() - start_t
        print(f"[AdaBN] Processed {min(i+1, n_adapt_batches)} batches in {adaptation_time:.2f}s ({adaptation_time*1000/n_adapt_batches:.1f} ms/batch).")

        # 4. Lock adapted statistics into eval mode
        self.model.eval()
        
        # 5. Record adapted target statistics
        for name, m in self.model.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                self.target_stats[name] = {
                    'mean': m.running_mean.clone().detach().cpu(),
                    'var': m.running_var.clone().detach().cpu()
                }
        print("[AdaBN] Adaptation complete! Target BatchNorm statistics locked for inference.")

    def compute_domain_drift_metrics(self):
        # Quantifies covariate shift: Layer-wise L2 norm of mean drift and variance drift
        drift_report = []
        for name in self.source_stats:
            src_m = self.source_stats[name]['mean']
            tgt_m = self.target_stats[name]['mean']
            src_v = self.source_stats[name]['var']
            tgt_v = self.target_stats[name]['var']
            
            mean_shift = torch.norm(src_m - tgt_m, p=2).item()
            var_shift  = torch.norm(src_v - tgt_v, p=2).item()
            drift_report.append({'layer': name, 'delta_mean': mean_shift, 'delta_var': var_shift})
        return drift_report


# Instantiate Adapter & Cache Pre-AdaBN Model State
adapter = AdaBNAdapter(model, device=config.device)

# Perform Adaptation
adapter.adapt_to_target_domain(target_unlabelled_loader, n_adapt_batches=config.n_adapt_batches)

# Inspect Sample Layer Statistics Shift
drift_report = adapter.compute_domain_drift_metrics()
print("\nSample Layer-Wise Domain Shift Quantification (Top Stages):")
sample_layers = [d for d in drift_report if any(k in d['layer'] for k in ['stem.1', 'layer1.0.bn1', 'layer2.0.bn1', 'layer3.0.bn1', 'layer4.0.bn1', 'rfb1', 'ra1'])]
for item in sample_layers[:8]:
    print(f"  * Layer: {item['layer']:25s} | Delta Mean: {item['delta_mean']:.4f} | Delta Var: {item['delta_var']:.4f}")


In [ ]:
# ==============================================================================
# CELL 8: CROSS-DOMAIN EVALUATION, DRIFT QUANTIFICATION & VISUAL OVERLAYS
# Pre-AdaBN vs Post-AdaBN Generalization Benchmark
# ==============================================================================

import pandas as pd
from scipy import stats

def detailed_evaluation(model, dataloader, device):
    # Computes DSC, mIoU, Precision, Recall, and Inference Latency
    model.eval()
    dices, ious, precs, recs = [], [], [], []
    latencies = []
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            start_t = time.time()
            with autocast():
                logits = model(imgs)
                if isinstance(logits, tuple):
                    logits = logits[0]
            latencies.append((time.time() - start_t) / imgs.size(0))
            
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            for i in range(len(preds)):
                p = preds[i, 0]
                g = masks[i, 0]
                tp = (p * g).sum().item()
                fp = (p * (1 - g)).sum().item()
                fn = ((1 - p) * g).sum().item()
                
                dice = (2.0 * tp + 1e-6) / (2.0 * tp + fp + fn + 1e-6)
                iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
                prec = (tp + 1e-6) / (tp + fp + 1e-6)
                rec  = (tp + 1e-6) / (tp + fn + 1e-6)
                
                dices.append(dice)
                ious.append(iou)
                precs.append(prec)
                recs.append(rec)
                
    fps = 1.0 / np.mean(latencies) if len(latencies) > 0 else 0
    return {
        'DSC': float(np.mean(dices)),
        'mIoU': float(np.mean(ious)),
        'Precision': float(np.mean(precs)),
        'Recall': float(np.mean(recs)),
        'FPS': float(fps),
        'raw_dices': dices
    }

# 1. Benchmark on Source Validation Set (In-Domain Reference)
print("\n[Evaluation 1/3] Benchmarking on Source Domain (Kvasir-SEG Val)...")
src_metrics = detailed_evaluation(model, source_val_loader, config.device)

# 2. Benchmark on Target Domain BEFORE AdaBN (Zero-Shot Direct Transfer)
# Re-inject source statistics to measure pre-adaptation baseline
for name, m in model.named_modules():
    if isinstance(m, nn.BatchNorm2d) and name in adapter.source_stats:
        m.running_mean.copy_(adapter.source_stats[name]['mean'].to(config.device))
        m.running_var.copy_(adapter.source_stats[name]['var'].to(config.device))

print("[Evaluation 2/3] Benchmarking on Target Domain BEFORE AdaBN (Direct Transfer)...")
pre_adabn_metrics = detailed_evaluation(model, target_test_loader, config.device)

# 3. Benchmark on Target Domain AFTER AdaBN (Adapted Transfer)
for name, m in model.named_modules():
    if isinstance(m, nn.BatchNorm2d) and name in adapter.target_stats:
        m.running_mean.copy_(adapter.target_stats[name]['mean'].to(config.device))
        m.running_var.copy_(adapter.target_stats[name]['var'].to(config.device))

print("[Evaluation 3/3] Benchmarking on Target Domain AFTER AdaBN (AdaBN Recalibrated)...")
post_adabn_metrics = detailed_evaluation(model, target_test_loader, config.device)

# Statistical Significance (Paired Wilcoxon test)
t_stat, p_val = stats.wilcoxon(post_adabn_metrics['raw_dices'], pre_adabn_metrics['raw_dices'])

# Tabulate Results
results_df = pd.DataFrame([
    {"Evaluation Setting": "Source Domain (In-Domain Reference)", "DSC (Dice)": f"{src_metrics['DSC']:.4f}", "mIoU": f"{src_metrics['mIoU']:.4f}", "Precision": f"{src_metrics['Precision']:.4f}", "Recall": f"{src_metrics['Recall']:.4f}", "FPS": f"{src_metrics['FPS']:.1f}"},
    {"Evaluation Setting": "Target Domain BEFORE AdaBN (Direct Transfer)", "DSC (Dice)": f"{pre_adabn_metrics['DSC']:.4f}", "mIoU": f"{pre_adabn_metrics['mIoU']:.4f}", "Precision": f"{pre_adabn_metrics['Precision']:.4f}", "Recall": f"{pre_adabn_metrics['Recall']:.4f}", "FPS": f"{pre_adabn_metrics['FPS']:.1f}"},
    {"Evaluation Setting": "Target Domain AFTER AdaBN (Domain Adapted)", "DSC (Dice)": f"{post_adabn_metrics['DSC']:.4f}", "mIoU": f"{post_adabn_metrics['mIoU']:.4f}", "Precision": f"{post_adabn_metrics['Precision']:.4f}", "Recall": f"{post_adabn_metrics['Recall']:.4f}", "FPS": f"{post_adabn_metrics['FPS']:.1f}"},
])

print("\n" + "="*85)
print("             ADABN-CHAKRANET CROSS-HOSPITAL GENERALIZATION BENCHMARK")
print("="*85)
print(results_df.to_string(index=False))
print("="*85)
delta_dsc = (post_adabn_metrics['DSC'] - pre_adabn_metrics['DSC']) * 100
delta_miou = (post_adabn_metrics['mIoU'] - pre_adabn_metrics['mIoU']) * 100
print(f"AdaBN Improvement: +{delta_dsc:.2f}% DSC | +{delta_miou:.2f}% mIoU (Wilcoxon p-value = {p_val:.4e})")

# Visual Prediction Overlays
print("\nRendering Cross-Hospital Visual Predictions Comparison...")
model.eval()
raw_imgs, gts, pre_preds, post_preds = [], [], [], []

with torch.no_grad():
    for imgs, masks in target_test_loader:
        imgs = imgs.to(config.device)
        
        # Pre-AdaBN
        for name, m in model.named_modules():
            if isinstance(m, nn.BatchNorm2d) and name in adapter.source_stats:
                m.running_mean.copy_(adapter.source_stats[name]['mean'].to(config.device))
                m.running_var.copy_(adapter.source_stats[name]['var'].to(config.device))
        pre_out = model(imgs)
        if isinstance(pre_out, tuple): pre_out = pre_out[0]
        pre_p = (torch.sigmoid(pre_out) > 0.5).cpu().numpy()
        
        # Post-AdaBN
        for name, m in model.named_modules():
            if isinstance(m, nn.BatchNorm2d) and name in adapter.target_stats:
                m.running_mean.copy_(adapter.target_stats[name]['mean'].to(config.device))
                m.running_var.copy_(adapter.target_stats[name]['var'].to(config.device))
        post_out = model(imgs)
        if isinstance(post_out, tuple): post_out = post_out[0]
        post_p = (torch.sigmoid(post_out) > 0.5).cpu().numpy()
        
        # De-normalize images for plotting
        mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
        std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
        img_np = np.clip((imgs.cpu().numpy() * std + mean) * 255, 0, 255).astype(np.uint8)
        
        for idx in range(min(4, len(imgs))):
            raw_imgs.append(np.transpose(img_np[idx], (1, 2, 0)))
            gts.append(masks[idx, 0].numpy())
            pre_preds.append(pre_p[idx, 0])
            post_preds.append(post_p[idx, 0])
        break

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
titles = ["Target Endoscopic Frame", "Ground Truth Mask", "Pre-AdaBN (Direct)", "Post-AdaBN (Adapted)"]

for row in range(min(4, len(raw_imgs))):
    axes[row, 0].imshow(raw_imgs[row])
    axes[row, 0].axis("off")
    if row == 0: axes[row, 0].set_title(titles[0], fontsize=13, fontweight='bold')

    axes[row, 1].imshow(gts[row], cmap="gray")
    axes[row, 1].axis("off")
    if row == 0: axes[row, 1].set_title(titles[1], fontsize=13, fontweight='bold')

    # Pre-AdaBN overlay (Red overlay on frame)
    pre_overlay = raw_imgs[row].copy()
    pre_overlay[pre_preds[row] > 0.5] = [255, 60, 60]
    axes[row, 2].imshow(cv2.addWeighted(raw_imgs[row], 0.6, pre_overlay, 0.4, 0))
    axes[row, 2].axis("off")
    if row == 0: axes[row, 2].set_title(titles[2], fontsize=13, fontweight='bold')

    # Post-AdaBN overlay (Green overlay on frame)
    post_overlay = raw_imgs[row].copy()
    post_overlay[post_preds[row] > 0.5] = [60, 255, 60]
    axes[row, 3].imshow(cv2.addWeighted(raw_imgs[row], 0.6, post_overlay, 0.4, 0))
    axes[row, 3].axis("off")
    if row == 0: axes[row, 3].set_title(titles[3], fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print("AdaBN-ChakraNet Pipeline Execution and Cross-Hospital Evaluation Complete.")
